# Fun & Fit 건강 어드바이저 에이전트 튜토리얼

**Fun & Fit 건강 어드바이저 에이전트** 튜토리얼에 오신 것을 환영합니다!  
이 튜토리얼에서는 **Microsoft Foundry** SDK를 사용하여  
재미있으면서도 (주의 문구가 포함된) 건강 & 피트니스 어시스턴트를 만들어봅니다.  

우리는 다음을 수행할 예정입니다:

1. **azure-ai-projects**를 사용하여 프로젝트를 **초기화**합니다.  
2. 전반적인 웰니스 및 영양 조언을 제공하는 (주의 문구 포함) **전문 에이전트 생성**  
3. 피트니스, 영양, 일반 건강 주제에 대한 **대화 관리**  
4. **OpenTelemetry**를 활용한 로깅 및 추적 **시연**  
5. 툴 활용, 주의 문구 삽입, 기본 모범 사례 적용을 **데모**합니다.

### ⚠️ 중요한 의료 주의 문구 ⚠️
> **이 노트북에서 제공하는 건강 정보는 일반적인 교육 및 오락 목적이며,  
> 전문적인 의학적 조언, 진단 또는 치료를 대체하기 위한 것이 아닙니다.**  
> 건강 상태와 관련된 질문이 있을 경우, 반드시 의사 또는 자격 있는 전문인의 조언을 구하세요.  
> 이 노트북에서 제공된 내용을 근거로 전문가의 조언을 무시하거나 치료를 지연하지 마십시오.

## 1. 초기 설정  
필요한 라이브러리를 임포트하고, 환경 변수를 불러오며,  
모든 에이전트 관련 작업을 수행할 수 있도록 **AIProjectClient**를 초기화합니다.  
그럼 시작해봅시다! 🎉


In [ ]:
# Import required libraries
import os  # For environment variables and path operations
import time  # For adding delays if needed
from pathlib import Path  # For cross-platform path handling

# Import Azure and utility libraries
from dotenv import load_dotenv  # For loading environment variables from .env file

from azure.identity import DefaultAzureCredential  # For Azure authentication
from azure.core.credentials import AzureKeyCredential

from azure.ai.projects import AIProjectClient  # Main client for AI Projects
from azure.ai.inference import ChatCompletionsClient

# Load environment variables
notebook_path = Path().absolute()
parent_dir = notebook_path.parent
load_dotenv(parent_dir / '.env')

# Load environment variables
azure_endpoint = os.environ.get("AZURE_ENDPOINT")
api_key = os.environ.get("API_KEY")

try:
    project_client = AIProjectClient(
        endpoint=azure_endpoint,
        credential=DefaultAzureCredential()
    )
    print("✅ Successfully initialized AIProjectClient")
except Exception as e:
    # Print error message if client initialization fails
    print(f"❌ Error initializing project client: {str(e)}")

## 2. Fun & Fit 건강 어드바이저 에이전트 만들기

전반적인 건강과 웰니스를 다루는 전문 에이전트를 생성해보겠습니다.  
에이전트의 지시문에는 **주의 문구**를 명시적으로 포함하여 항상 안전을 우선하도록 합니다.  
또한 에이전트가 **일반적인 피트니스**, **식단 팁**에 집중하고  
사용자에게 **전문가의 조언을 구하도록 권장**하는 역할을 수행하도록 지시할 것입니다.


In [ ]:
def create_health_advisor_agent():
    model_name = os.environ.get("MODEL_NAME")
    
    """Create a health advisor agent with disclaimers and basic instructions."""
    try:
        # Create a new agent using the AIProjectClient
        # The agent will use the specified model and follow the given instructions
        agent = project_client.agents.create_agent(
            model=model_name,
            name="fun-fit-health-advisor",
            # Define the agent's behavior and responsibilities
            instructions="""
            You are a friendly AI Health Advisor.
            You provide general health, fitness, and nutrition information, but always:
            1. Include medical disclaimers.
            2. Encourage the user to consult healthcare professionals.
            3. Provide general, non-diagnostic advice around wellness, diet, and fitness.
            4. Clearly remind them you're not a doctor.
            5. Encourage safe and balanced approaches to exercise and nutrition.
            """
        )
        # Log success and return the created agent
        print(f"🎉 Created health advisor agent, ID: {agent.id}")
        return agent
    except Exception as e:
        # Handle any errors during agent creation
        # print(f"❌ Error creating agent: {str(e)}")
        print(f"❌ Error creating agent:", e)
        return None

# Create an instance of our health advisor agent
health_advisor = create_health_advisor_agent()

## 3. 건강 대화 관리하기 
대화(또는 *스레드*)는 사용자의 메시지와 에이전트의 건강 관련 응답을 저장하는 공간입니다.  
**건강 & 피트니스 Q&A**에 전용된 새로운 스레드를 만들어봅시다.


In [ ]:
# Function to create a new conversation thread for health discussions
def start_health_conversation():
    """Create a new thread for health & fitness discussions."""
    try:
        # Create a thread for communication
        thread = project_client.agents.threads.create()
        print(f"Created thread, ID: {thread.id}")
        
        # Return the created thread object so we can use it later
        return thread
    except Exception as e:
        # If thread creation fails, print error and return None
        print(f"❌ Error creating thread: {str(e)}")
        return None

# Initialize a new conversation thread that we'll use for our health Q&A
health_thread = start_health_conversation()

## 4. 건강 & 피트니스 질문하기 
사용자로부터 일반적인 건강 질문을 담은 메시지를 만들어보겠습니다.  
예를 들어, **"BMI는 어떻게 계산하나요?"** 또는 **"활동적인 라이프스타일을 위한 균형 잡힌 식단은 무엇인가요?"** 같은 질문이 될 수 있습니다.  

우리의 건강 어드바이저 에이전트가 이 질문들에 응답하게 하되,  
항상 **주의 문구**를 잊지 않도록 할 것입니다!


In [ ]:
def ask_health_question(thread_id, user_question):
    """Add a user message to the conversation thread.
    
    Args:
        thread_id: ID of the conversation thread
        user_question: The health/fitness question from the user
        
    Returns:
        Message object if successful, None if error occurs
    """
    try:
        # Create a new message in the thread from the user's perspective
        # The role="user" indicates this is a user message (vs assistant)
        return project_client.agents.messages.create(
            thread_id=thread_id,
            role="user", 
            content=user_question
        )
    except Exception as e:
        print(f"❌ Error adding user message: {e}")
        return None

def process_thread_run(thread_id, agent_id):
    """Ask the agent to process the thread and generate a response.
    
    Args:
        thread_id: ID of the conversation thread
        agent_id: ID of the health advisor agent
        
    Returns:
        Run object if successful, None if error occurs
    """
    try:
        # Create a new run to have the agent process the thread
        run = project_client.agents.runs.create_and_process(
            thread_id=thread_id,
            agent_id=agent_id
        )

        # Poll the run status until completion or error
        # Status can be: queued, in_progress, requires_action, completed, failed
        while run.status in ["queued", "in_progress", "requires_action"]:
            time.sleep(1)  # Wait 1 second between status checks
            run = project_client.agents.get_run(
                thread_id=thread_id,
                run_id=run.id
            )

        print(f"🤖 Run completed with status: {run.status}")
        return run
    except Exception as e:
        print(f"❌ Error processing thread run: {str(e)}")
        return None

def view_thread_messages(thread_id):
    """Display all messages in the conversation thread in chronological order.
    
    Args:
        thread_id: ID of the conversation thread to display
    """
    try:
        # Fetch and log all messages
        messages = project_client.agents.messages.list(thread_id=thread_id)
        for message in messages:
            print(f"Role: {message.role}, Content: {message.content}")
    except Exception as e:
        print(f"❌ Error viewing thread: {str(e)}")

### 예시 질문들  
에이전트가 어떻게 **주의 문구**를 포함하고, 일반적인 건강 질문에 어떻게 응답하는지 확인해보기 위해  
간단한 질의들을 시도해보겠습니다.  

이번에는 **BMI(체질량지수)** 와 **활동적인 라이프스타일에 적합한 균형 잡힌 식단**에 대해 질문해볼 예정입니다.


In [ ]:
# First verify that we have valid agent and thread objects before proceeding
if health_advisor and health_thread:
    # 1) Ask about BMI calculation and interpretation
    # This demonstrates how the agent handles technical health questions
    msg1 = ask_health_question(health_thread.id, "How do I calculate my BMI, and what does it mean?")
    # Process the BMI question and wait for agent's response
    run1 = process_thread_run(health_thread.id, health_advisor.id)

    # 2) Ask about personalized meal planning
    # This shows how the agent provides customized nutrition advice
    msg2 = ask_health_question(health_thread.id, "Can you give me a balanced meal plan for someone who exercises 3x a week?")
    # Process the meal plan question and wait for agent's response
    run2 = process_thread_run(health_thread.id, health_advisor.id)

    # Display the full conversation history to see both Q&As
    # This will show the agent's responses including any health disclaimers
    view_thread_messages(health_thread.id)
else:
    # Error handling if agent or thread initialization failed
    print("❌ Could not run example queries because agent or thread is None.")

## 5. 정리하기 🧹  
작업이 완료된 후 에이전트를 서비스에서 제거하고 싶다면 아래에서 삭제할 수 있습니다.  
(운영 환경에서는 **상태 유지 기반의 경험**을 위해 에이전트를 계속 유지할 수도 있습니다!)


In [ ]:
# Function to clean up and delete the agent when we're done
def cleanup(agent):
    # Only attempt cleanup if we have a valid agent
    if agent:
        try:
            # Delete the agent using the project client
            project_client.agents.delete_agent(agent.id)
            # Print confirmation message with the agent name
            print(f"🗑️ Deleted health advisor agent: {agent.name}")
        except Exception as e:
            # Handle any errors that occur during deletion
            print(f"Error cleaning up agent: {e}")
    else:
        # If no agent was provided, inform the user
        print("No agent to clean up.")

# Call cleanup function to delete our health advisor agent
cleanup(health_advisor)

# 축하합니다! 
여러분은 성공적으로 **Fun & Fit 건강 어드바이저**를 구축하셨습니다!  
이 에이전트는 다음과 같은 기능을 수행할 수 있습니다:

1. 기본적인 건강 및 피트니스 질문에 **응답**합니다.  
2. **주의 문구**를 활용하여 안전하고 전문적인 상담을 권장합니다.  
3. 일반적인 식단 및 웰니스 정보를 **제공**합니다.  
4. **Microsoft Foundry** 모듈의 시너지를 활용하여 대화를 **구성**합니다.

건강한 코딩 하세요!
